ZeRO’s main concept is to `leverage memory redundancy` in **data-parallel training** and enhance `inter-GPU communication` efficiency for improved throughput, with a trade-off of` increased communication volume depending` on the stage. ZeRO consists of two key components:

-   ZeRO-DP (Data Parallelism)
-   ZeRO-R (Residual Memory)

![](https://archive.is/WRg2M/c89877c155a2204041e2406c5f254baccb16ac50.webp)

Figure 2: source:  [https://www.microsoft.com/en-us/research/blog/zero-deepspeed-new-system-optimizations-enable-training-models-with-over-100-billion-parameters/](https://archive.is/o/WRg2M/https://www.microsoft.com/en-us/research/blog/zero-deepspeed-new-system-optimizations-enable-training-models-with-over-100-billion-parameters/)

As depicted in Figure , ZeRO has three main optimisation stages, which correspond to the partitioning of optimiser states, gradients, and parameters. When enabled cumulatively:

_1. **Optimizer State** Partitioning (Pos) — 4x memory reduction, same communication volume as data parallelism_

_2. **Add Gradient Partitioning** (Pos+g) — 8x memory reduction, same communication volume as data parallelism_

_3. Add **Parameter Partitioning** (Pos+g+p) — Memory reduction is linear with data parallelism degree Nd._

ZeRO eliminates memory redundancies and makes the full aggregate memory capacity of a cluster available. The DeepSpeed team has also put out more compelling —  **ZeRO-Offload**/Infinity (offloading computation to CPU/ NVMe disk) and ZeRO++ (with flexible multi-node training and quantized weights).

ZeRO (Zero Redundancy Optimizer) is a breakthrough approach that tackles the key inefficiencies in distributed deep learning by addressing the memory overhead inherent in traditional data parallelism and the intricate dependencies typical of model parallelism. Instead of replicating all model states (parameters, gradients, and optimizer states) across every process, ZeRO smartly partitions these states across devices. This design allows for the efficient training of very large models without the crippling memory redundancy or communication bottlenecks.

Below is a detailed breakdown of how ZeRO overcomes the limitations and what each optimization stage entails:

----------

### 1. The Challenge with Traditional Parallelism

-   **Data Parallelism:**
    
    -   **Pros:** Easy to scale; each GPU has a copy of the complete model and processes a different mini-batch of data.
    -   **Cons:** Every GPU holds a full replica of model parameters, gradients, and optimizer states. This replication leads to a massive memory overhead—a severe constraint when training extremely large models.
-   **Model Parallelism:**
    
    -   **Pros:** Splits model parameters across GPUs, reducing per-device memory usage.
    -   **Cons:** Often suffers from complex inter-device communications and fine-grained partitioning, which can hamper computational efficiency.

----------



----------

### 3. Stages of ZeRO Optimization

#### **Stage 1: Partitioning Optimizer States**

-   **What Happens:**  
    Optimizer states (for instance, in the Adam optimizer, these include the 32-bit weights along with the first and second moment estimates) are partitioned so that each process holds only a fraction of these states.
    
-   **Benefit:**  
    Each GPU only maintains and updates its assigned slice of the optimizer state, drastically reducing redundant memory usage across processes.
    

#### **Stage 2: Partitioning Gradients**

-   **What Happens:**  
    Instead of each device retaining the entire 32-bit gradient for all parameters, ZeRO partitions the gradients. Each process only keeps the gradients corresponding to its portion of the optimizer state.
    
-   **Benefit:**  
    This step further cuts down on memory consumption and ensures that gradient computations are distributed efficiently across devices.
    

#### **Stage 3: Partitioning Parameters**

-   **What Happens:**  
    ZeRO-3 takes the partitioning a step further by dividing the 16-bit model parameters themselves across devices. During the forward and backward passes, these parameters are collected and partitioned on the fly.
    
-   **Benefit:**  
    The model’s weight matrix, typically one of the largest components of memory use, no longer needs to be fully replicated. This enables the training of models that would otherwise be too large to fit into a single GPU’s memory.
    

----------

### 4. ZeRO-Infinity: Beyond GPUs

-   **Infinity Offload Engine:**  
    ZeRO-3 has evolved to include an infinity offload engine, leading to what is known as **ZeRO-Infinity.** This feature further extends the capabilities by offloading parts of the model state not just to the CPU but also to NVMe storage.
    
-   **Benefit:**  
    Offloading to both CPU and NVMe allows for even greater memory savings, meaning models that exceed GPU memory limits can still be trained efficiently. This is particularly vital when working with extremely large models or in environments with limited GPU resources.
    

----------

### 5. Dynamic Communication Scheduling

-   **Role in ZeRO:**  
    Although model states are partitioned, ZeRO isn’t just about slicing up memory. It also incorporates a dynamic communication schedule that ensures the necessary state information is shared across devices when needed during training.
    
-   **Impact:**  
    This smart scheduling keeps the computational granularity and communication volume comparable to what you would expect from data parallelism. Essentially, it minimizes the communication overhead while still reaping the memory benefits of partitioning.
 

| Setup                   | Scenario                                      | Strategy                                                                           |
|-------------------------|-----------------------------------------------|------------------------------------------------------------------------------------|
| Single Node / Multi-GPU | Fits on single GPU                            | DistributedDataParallel or ZeRO                                                   |
| Single Node / Multi-GPU | Doesn’t fit on single GPU                     | PipelineParallel, ZeRO, or TensorParallel                                          |
| Single Node / Multi-GPU | Largest model layer doesn’t fit               | TensorParallel or ZeRO                                                             |
| Multi-node / Multi-GPU  | Fast inter-node connectivity (NVLink/NVSwitch)  | ZeRO or 3D parallelism (PipelineParallel, TensorParallel, DataParallel)            |
| Multi-node / Multi-GPU  | Slow inter-node connectivity                  | ZeRO or 3D parallelism (PipelineParallel, TensorParallel, DataParallel)            |

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/parallelism-deepspeed-3d.png)

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/parallelism-zero-dp-pp.png)
-  **F0, F1, F2, F3 (Forward Pass):**

- Here, **"F" stands for "Forward"**—the part where you process data through the layers.

- The number typically indicates *either* the order of the micro-batch or the stage at which it is processed in forward order.

-  **For example:**

-  **F0:** The first micro-batch begins processing at Stage 0 (the first station).

-  **F1:** As soon as Stage 0 finishes with F0, it can start processing the next micro-batch, F1, or F0 moves to Stage 1 while F1 begins at Stage 0.

- This staggered processing continues until the last stage gets its micro-batch, say F3 at Stage 3.

  

-  **B3, B2, B1, B0 (Backward Pass):**

- Here, **"B" stands for "Backward"**—this is when gradients (or errors) are calculated and sent back through the network.

- The numbering goes **in reverse** relative to the forward pass because the last stage starts the backward computation first.

-  **For example:**

-  **B3:** Once micro-batch F3 has finished the forward pass (at the last stage), it immediately starts the backward pass and is labeled B3.

-  **B2:** Then, the process continues backward: the output of Stage 2 gets processed as B2, and so on until B0 at Stage 0.
Stage 0      Stage 1      Stage 2      Stage 3
  |            
F0 ->        F1 ->         F2 ->         F3 ->
  |            
  | <--B0     | <--B1     | <--B2     | <--B3

-  **Multiple micro-batches are processed concurrently.**

-  **The idle time (bubbles) is minimized by careful scheduling.**

-  **The benefits of overlapping computation across different stages often outweigh the losses due to those bubbles.**